# Aula 1 — Fundamentos de HTTP e REST (exemplos práticos)

Para ver requests e responses HTTP de verdade sem depender de internet,
vamos criar um servidor bem simples localmente (usando só a biblioteca
padrão) e fazer pedidos reais a ele com a biblioteca `requests`.

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

tarefas = {1: {"id": 1, "titulo": "Estudar Python", "concluida": False}}

class ServidorTarefas(BaseHTTPRequestHandler):
    def _responder_json(self, status, corpo):
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(corpo).encode("utf-8"))

    def do_GET(self):
        if self.path == "/tarefas":
            self._responder_json(200, list(tarefas.values()))
        else:
            self._responder_json(404, {"erro": "não encontrado"})

    def log_message(self, format, *args):
        pass  # silencia o log padrão do servidor no notebook

servidor = HTTPServer(("127.0.0.1", 0), ServidorTarefas)
porta = servidor.server_address[1]

thread_servidor = threading.Thread(target=servidor.serve_forever, daemon=True)
thread_servidor.start()

print(f"Servidor local rodando em http://127.0.0.1:{porta}")

## Fazendo um pedido `GET` de verdade

In [ ]:
import requests

resposta = requests.get(f"http://127.0.0.1:{porta}/tarefas")

print("status code:", resposta.status_code)
print("headers:", dict(resposta.headers))
print("corpo (JSON):", resposta.json())

## Um caminho que não existe: `404`

In [ ]:
resposta = requests.get(f"http://127.0.0.1:{porta}/nao-existe")
print("status code:", resposta.status_code)
print("corpo:", resposta.json())

## Experimento guiado

Adicione uma segunda tarefa ao dicionário `tarefas` e rode a célula do
`GET /tarefas` de novo -- confirme que a nova tarefa aparece na
resposta.

## Mini-desafio resolvido

**Desafio:** medir quanto tempo o pedido HTTP demora, usando os próprios recursos da biblioteca `requests`.

In [ ]:
resposta = requests.get(f"http://127.0.0.1:{porta}/tarefas")
print(f"O pedido levou {resposta.elapsed.total_seconds() * 1000:.2f} ms")

## Encerrando o servidor local

In [ ]:
servidor.shutdown()
servidor.server_close()
print("Servidor local encerrado.")